# 08 - Feature Extraction Spatial

In [ ]:
import os, sys, zipfile
from pathlib import Path

# Cari root repo: di laptop = folder di atas notebooks/, di Kaggle = dataset mlc-src
if os.path.exists("/kaggle/input"):
    hits = list(Path("/kaggle/input").rglob("src/data.py"))
    if hits:
        ROOT = hits[0].parent.parent
    else:  # dataset masih berupa zip
        ROOT = Path("/kaggle/working/_repo")
        for z in Path("/kaggle/input").rglob("*.zip"):
            if z.stem in ("src", "configs", "annotations"):
                zipfile.ZipFile(z).extractall(ROOT / z.stem)
else:
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.data import ON_KAGGLE, get_path, load_config
from src.seed import seed_everything

cfg = load_config()
seed_everything(cfg["SEED"])
print("Kaggle:", ON_KAGGLE, "| data:", get_path(cfg, "raw"))

Fitur per frame yang **mempertahankan letak**: token patch DINOv2 (kisi 18 x 32 untuk input
252 x 448) dikumpulkan ke kisi kasar, bukan dirata-rata jadi satu angka. Tujuannya supaya detail
kecil (tangan, jam tangan, kertas) tidak hilang seperti pada rata-rata global notebook 03.

Keluaran per frame: `[1 + gh*gw, 384]` dengan token 0 = CLS dan sel kisi urut baris demi baris.
Praproses frame, seed per video, dan JPEG bolak-balik **identik dengan notebook 03** (FAQ 73);
yang berbeda hanya cara token patch diringkas. Test: semua 30 frame, satu sequence demi satu
sequence, tanpa statistik apa pun dari data test (FAQ 38, 66, 68, 70).

In [ ]:
import json
import time

import numpy as np
import torch

from src import features as F

ENCODER = "vit_small_patch14_dinov2.lvd142m"
SEED = cfg["SEED"]
SEQ_LEN = cfg["SEQ_LEN"]
BATAS_GB = 6.0                  # di atas ini kisi diperkecil supaya hasil tidak kebesaran
GRID = (3, 8)                   # pilihan awal; bisa turun ke (3, 4) setelah ukuran dihitung
BATCH = 64

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Di Kaggle tanpa GPU, ekstraksi bisa makan berjam-jam. Lebih baik berhenti dan perbaiki setelan.
assert DEVICE == "cuda" or not ON_KAGGLE, (
    "GPU belum aktif. Jalankan lewat kaggle_run.py tanpa --cpu, dan pastikan akun sudah verifikasi telepon."
)
print("device:", DEVICE, "| encoder:", ENCODER, "| kisi patch:", F.TINGGI_PATCH, "x", F.LEBAR_PATCH)

raw = get_path(cfg, "raw")
out_dir = get_path(cfg, "features")          # ditulis ke /kaggle/working/features di Kaggle
emb_sp_train_dir = out_dir / "emb_sp_train"
print("fitur disimpan ke:", out_dir)

## Daftar video + perkiraan ukuran keluaran (penentu kisi)

In [ ]:
# Nama file keluaran: <kategori>__<subject>__<nama video>.npy (sama dengan notebook 03)
def nama_fitur(p):
    p = Path(p)
    return f"{p.parent.parent.name}__{p.parent.name}__{p.stem}"


videos = sorted(
    p for kategori in ("normal", "cheat")
    for p in (raw / "train" / kategori).glob("subject_*/*") if p.suffix.lower() == ".mov"
)
print("jumlah video train:", len(videos))
assert len(videos) > 0, "Tidak ada video train ditemukan"

folder_test = sorted(p for p in (raw / "test").iterdir() if p.is_dir())
print("jumlah sequence test:", len(folder_test))


def jumlah_frame(path):
    import cv2

    cap = cv2.VideoCapture(str(path))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return n


t0 = time.time()
n_frame = [jumlah_frame(p) for p in videos]
FRAME_TRAIN = int(sum(n_frame))
print(f"total frame train: {FRAME_TRAIN} (min {min(n_frame)}, maks {max(n_frame)}) "
      f"- dibaca dari metadata dalam {time.time() - t0:.0f} s")


def perkiraan_gb(grid, dim=384):
    n_token = 1 + grid[0] * grid[1]
    byte = (FRAME_TRAIN + len(folder_test) * SEQ_LEN) * n_token * dim * 2   # float16
    return byte / 1e9, n_token


for kandidat in ((3, 8), (3, 4)):
    gb, n_token = perkiraan_gb(kandidat)
    print(f"  kisi {kandidat}: {n_token} token/frame -> perkiraan {gb:.2f} GB")

gb, _ = perkiraan_gb(GRID)
if gb > BATAS_GB:
    GRID = (3, 4)
    gb, _ = perkiraan_gb(GRID)
    print(f"LAPOR: perkiraan di atas {BATAS_GB} GB, kisi diperkecil ke {GRID} -> {gb:.2f} GB")
else:
    print(f"kisi dipakai: {GRID} -> perkiraan {gb:.2f} GB (di bawah batas {BATAS_GB} GB)")

## Encoder beku + perkiraan waktu

In [ ]:
enc = F.buat_encoder(ENCODER, device=DEVICE, pretrained=True)
N_TOKEN, DIM = F.bentuk_spasial(enc, GRID)
print("bentuk fitur per frame:", (N_TOKEN, DIM), "| token 0 = CLS, sisanya sel kisi baris demi baris")

# Ukur kecepatan dengan frame tiruan (hanya untuk perkiraan waktu, tidak menyentuh data apa pun)
uji = [np.zeros((F.TINGGI_ASAL, F.LEBAR_ASAL, 3), dtype=np.uint8) for _ in range(BATCH)]
F.embed_frames_spatial(enc, uji[:8], DEVICE, GRID, batch=8)        # pemanasan
t0 = time.time()
F.embed_frames_spatial(enc, uji, DEVICE, GRID, batch=BATCH)
laju = BATCH / (time.time() - t0)
total_frame = FRAME_TRAIN + len(folder_test) * SEQ_LEN
print(f"laju kasar: {laju:.0f} frame/s -> perkiraan {total_frame / laju / 60:.0f} menit "
      f"untuk {total_frame} frame (train + test)")

## Fitur train (bisa dilanjutkan)

Hasil tiap video langsung ditulis ke `.npy` lewat berkas sementara lalu diganti nama, jadi kalau
sesi terputus berkas yang ada tidak pernah separuh dan video yang sudah jadi akan dilewati.

In [ ]:
t0 = time.time()
F.ekstrak_banyak_video_spatial(enc, videos, emb_sp_train_dir, DEVICE, grid=GRID,
                               seed=SEED, batch=BATCH, nama_fn=nama_fitur)
print(f"selesai video train dalam {(time.time() - t0) / 60:.1f} menit")

## Fitur test (bisa dilanjutkan)

Ditulis langsung ke `emb_sp_test.npy` lewat memmap; kemajuannya dicatat di berkas `.progress`
sehingga ekstraksi lanjut dari sequence terakhir, bukan dari awal.

In [ ]:
emb_sp_test_path = out_dir / "emb_sp_test.npy"
ids_sp_test_path = out_dir / "emb_sp_test_ids.txt"

t0 = time.time()
ids, feats = F.ekstrak_test_spatial(enc, raw / "test", emb_sp_test_path, DEVICE,
                                    grid=GRID, batch=SEQ_LEN)
ids_sp_test_path.write_text("\n".join(ids), encoding="utf-8")
print(f"selesai test dalam {(time.time() - t0) / 60:.1f} menit | {feats.shape}")
del feats

## Pemeriksaan hasil + meta.json + ukuran total

In [ ]:
npy = sorted(emb_sp_train_dir.glob("*.npy"))
assert len(npy) == len(videos), f"{len(npy)} berkas fitur vs {len(videos)} video"
bentuk = [np.load(p, mmap_mode="r").shape for p in npy]
assert all(b[1:] == (N_TOKEN, DIM) for b in bentuk), f"bentuk tak seragam: {set(b[1:] for b in bentuk)}"
panjang = [b[0] for b in bentuk]
print("video train berfitur:", len(npy), "| frame min/maks:", min(panjang), max(panjang),
      "| bentuk per video: [T,", N_TOKEN, ",", DIM, "]")
assert np.isfinite(np.load(npy[0]).astype(np.float32)).all(), "Ada NaN/inf di fitur train"

et = np.load(emb_sp_test_path, mmap_mode="r")
ids = ids_sp_test_path.read_text(encoding="utf-8").split("\n")
print("test:", et.shape, "| jumlah id:", len(ids))
assert et.shape == (len(folder_test), SEQ_LEN, N_TOKEN, DIM), et.shape
assert len(ids) == et.shape[0], f"{len(ids)} id vs {et.shape[0]} sequence"
assert np.isfinite(np.asarray(et[:20], dtype=np.float32)).all(), "Ada NaN/inf di fitur test"

byte_train = sum(p.stat().st_size for p in npy)
byte_test = emb_sp_test_path.stat().st_size
print(f"ukuran: train {byte_train / 1e9:.2f} GB | test {byte_test / 1e9:.2f} GB | "
      f"total {(byte_train + byte_test) / 1e9:.2f} GB")

meta = {"encoder": ENCODER, "dim": int(DIM), "grid": list(GRID), "n_token": int(N_TOKEN),
        "seed": SEED, "kualitas_jpeg": list(F.KUALITAS_JPEG),
        "ukuran_input": [F.TINGGI_INPUT, F.LEBAR_INPUT],
        "kisi_patch": [F.TINGGI_PATCH, F.LEBAR_PATCH],
        "n_video_train": len(npy), "frame_train": int(sum(panjang)),
        "n_seq_test": int(et.shape[0]), "bentuk_test": list(et.shape),
        "ukuran_gb": round((byte_train + byte_test) / 1e9, 3),
        "catatan": "token 0 = CLS; sel kisi urut baris demi baris; praproses identik notebook 03"}
(out_dir / "meta_spasial.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print(meta)